### Twist – Lösung Schritt für Schritt

*37. Bundeswettbewerb Informatik (2018), 1. Runde, Aufgabe 2*

Elvis hat vor einiger Zeit einen interessanten Artikel gelesen: Wenn in einem Text die Buchstaben jedes Wortes zufällig umgeordnet werden, der erste und letzte Buchstabe aber beibehalten werden, dann können Menschen den Text oft trotzdem noch lesen. Elvis, ein begnadeter Tänzer, nennt dieses Umordnen „Twisten“. Zum Beispiel:

    Der Twsit
    (Eigsnclh tiwst = Duenrhg, Venurdrehg)
    war ein Mdaotenz im 4/4-Tkat,
    der in den frhüen 1960er Jearhn populär
    wrude und zu
    Rcok'n'Roll, Ryhthm and Bleus oedr sielezpler
    Twsit-Msuik gnzteat wrid.

Elvis hat eine Idee: Um seine privaten Nachrichten vor Mitlesen durch Maschinen zu sichern, möchte er sie „twisten“. Denn: Menschen können seine Nachrichten trotzdem noch lesen, aber Maschinen werden in die Irre geführt.

Jerry kann das kaum glauben: wenn ein Mensch einen Text „enttwisten“ kann, so doch sicher auch eine Maschine. Oder?

#### Aufgabe 2

- Hilf Elvis und schreibe ein Programm, das einen Text twistet.
- Überlege dir, wie ein getwisteter Text enttwistet werden könnte. Hilf Jerry und schreibe ein Programm, das einen gegebenen Text mit (getwisteten) deutschen Wörtern möglichst gut enttwistet. Wie gut schafft das dein Programm?

Wende dein Programm auf das obige Beispiel an und auf alle weiteren Beispiele, die du auf den BWINF-Webseiten findest. Du findest dort auch eine Liste mit den Wörtern der deutschen Sprache.

#### Dateien (`beispieldaten/`)

- `twist1.txt` bis `twist5.txt`: Texte zum Twisten (`twist1.txt` ist das Beispiel oben vor dem Twisten)
- `enttwist.txt`: das getwistete Beispiel zum Enttwisten
- `woerterliste.txt`: lange Liste deutscher Wörter, ein Wort pro Zeile (312 002 Einträge)

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zufälliges Mischen (Fisher-Yates):** Man geht die Positionen von hinten nach vorn durch und tauscht jede mit einer zufälligen Position davor (oder mit sich selbst). So ist jede Anordnung gleich wahrscheinlich, und man braucht keine zweite Liste.
- **Normalform (Schlüssel):** Wenn viele Dinge „gleichwertig“ sind (hier: durch Twisten auseinander hervorgegangen), berechnet man für jedes Ding eine Normalform. Gleichwertige Dinge haben dieselbe Normalform. Statt zu vergleichen, schlägt man die Normalform nach.
- **Nachschlagen im Wörterbuch (`dict`):** Mit einem Schlüssel findet man den Eintrag fast sofort, ohne die ganze Liste zu durchsuchen.

**Nützliche Funktionen und Bibliotheken**
- `random.randint(a, b)` für Zufallszahlen von `a` bis `b` (beide eingeschlossen), `random.seed(...)` für reproduzierbare Läufe.
- `sorted(...)` sortiert die Zeichen eines Wortes, `''.join(liste)` setzt Zeichen wieder zu einem Wort zusammen.
- `z.isalpha()` fragt, ob ein Zeichen ein Buchstabe ist (auch ä, ö, ü, ß); `z.isupper()` fragt nach einem Großbuchstaben.
- `dict.get(schluessel, [])` liefert den Eintrag oder einen Ersatzwert, wenn es den Schlüssel nicht gibt.

## Die Idee

**Twisten.** Ein Wort besteht aus erstem Buchstaben, *inneren* Buchstaben und letztem Buchstaben. Nur die inneren Buchstaben werden gemischt. Wörter mit höchstens 3 Buchstaben haben höchstens einen inneren Buchstaben und bleiben, wie sie sind. Zahlen, Satzzeichen und Leerzeichen sind keine Wörter und bleiben stehen.

**Enttwisten.** Ein getwistetes Wort hat mit dem Originalwort gemeinsam:

1. den gleichen ersten und letzten Buchstaben,
2. die gleiche Länge und die gleichen inneren Buchstaben (nur in anderer Reihenfolge).

Alle Anordnungen der inneren Buchstaben durchzuprobieren ist hoffnungslos: Schon bei „Informatiker“ gibt es für die 10 inneren Buchstaben `10!` Anordnungen, also über 3,6 Millionen, bei längeren Wörtern noch viel mehr. Besser ist die **Normalform**: Wir sortieren die inneren Buchstaben. „Programm“ und „Prarmgom“ haben beide die Normalform `p` + `agmorrm` + `m`. Alle Wörter, die durch Twisten ineinander übergehen, haben dieselbe Normalform. Also:

1. Für jedes Wort der Wörterliste die Normalform berechnen und in einem Wörterbuch (`dict`) ablegen: Normalform → Liste der Wörter.
2. Für ein getwistetes Wort die Normalform berechnen und nachschlagen.

Dabei können drei Dinge passieren: genau ein Treffer (gut), mehrere Treffer (mehrdeutig, z. B. „frhüen“ kann „frühen“ oder „führen“ sein) oder gar kein Treffer (das Wort steht nicht in der Liste).

## Schritt 1: Text in Wörter und Reste zerlegen

Wir zerlegen den Text in Stücke: abwechselnd Wörter (nur Buchstaben) und Rest (Leerzeichen, Ziffern, Satzzeichen). Jedes Stück bekommt die Marke `True` (Wort) oder `False` (Rest). Das Zusammenfügen aller Stücke ergibt wieder genau den Text. Das Zeichen `'` in „Rock'n'Roll“ gehört zum Rest, deshalb sind „Rock“, „n“ und „Roll“ drei Wörter.

In [ ]:
def zerlege(text):
    """Gibt eine Liste von Paaren (ist_wort, stueck) zurück; zusammengesetzt ergeben die Stücke wieder den Text."""
    stuecke = []
    aktuell = ''                       # das Stück, das gerade aufgebaut wird
    im_wort = False                    # ist das aktuelle Stück ein Wort?
    for z in text:
        if z.isalpha() == im_wort:     # gleiche Sorte wie das aktuelle Stück: anhängen
            aktuell = aktuell + z
        else:                          # Sortenwechsel: Stück abschließen, neues beginnen
            if aktuell != '':
                stuecke.append((im_wort, aktuell))
            aktuell = z
            im_wort = z.isalpha()
    if aktuell != '':
        stuecke.append((im_wort, aktuell))
    return stuecke


def lies_text(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read()


text1 = lies_text('beispieldaten/twist1.txt')
print(zerlege("Rock'n'Roll, 4/4-Takt")[:8])
print('Rückprobe:', ''.join(s for (w, s) in zerlege(text1)) == text1)

[(True, 'Rock'), (False, "'"), (True, 'n'), (False, "'"), (True, 'Roll'), (False, ', 4/4-'), (True, 'Takt')]
Rückprobe: True


## Schritt 2: Ein Wort twisten (Fisher-Yates)

Die inneren Buchstaben kommen in eine Liste. Dann gehen wir die Positionen von hinten nach vorn durch (Position `i` von `len - 1` bis `1`), wählen eine zufällige Position `j` von `0` bis `i` und tauschen die Buchstaben an `i` und `j`. Das ist der **Fisher-Yates-Shuffle**: Er arbeitet in der Liste selbst und braucht nur `n - 1` Tauschschritte. Der einfachere Weg („zufälligen Buchstaben aus einer Liste ziehen und dort entfernen“) wäre bei langen Listen langsamer, weil das Entfernen aus der Mitte alle folgenden Elemente verschiebt.

Wörter mit höchstens 3 Buchstaben lassen wir unverändert. Bei 4 Buchstaben gibt es nur zwei innere Buchstaben: Mit Wahrscheinlichkeit 1/2 bleibt das Wort zufällig, wie es ist, das ist erlaubt.

In [ ]:
import random


def twiste_wort(wort):
    """Mischt die inneren Buchstaben eines Wortes; erster und letzter Buchstabe bleiben."""
    if len(wort) <= 3:
        return wort
    innen = list(wort[1:-1])
    for i in range(len(innen) - 1, 0, -1):
        j = random.randint(0, i)                      # beide Grenzen eingeschlossen
        innen[i], innen[j] = innen[j], innen[i]       # vertauschen
    return wort[0] + ''.join(innen) + wort[-1]


random.seed(37)
for wort in ['Programm', 'Informatiker', 'Twist', 'der', 'Takt']:
    print(wort, '->', twiste_wort(wort))

Programm -> Porgramm
Informatiker -> Intkomrefiar
Twist -> Tswit
der -> der
Takt -> Tkat


## Schritt 3: Einen ganzen Text twisten

Wir zerlegen den Text, twisten jedes Wort-Stück und setzen alles wieder zusammen. Rest-Stücke (Satzzeichen, Zahlen, Leerzeichen, Zeilenumbrüche) bleiben unverändert.

In [ ]:
def twiste_text(text):
    ergebnis = ''
    for ist_wort, stueck in zerlege(text):
        if ist_wort:
            ergebnis = ergebnis + twiste_wort(stueck)
        else:
            ergebnis = ergebnis + stueck
    return ergebnis


random.seed(37)
print(twiste_text(text1))

Der Tiwst
(Eiglncsh tswit = Drnehug, Vduerherng)
war ein Mdoantez im 4/4-Tkat,
der in den führen 1960er Jehran puläpor
wdure und zu
Rcok'n'Rlol, Rhhtym and Bules oder seleielzpr
Tswit-Muisk gantzet wrid.


Das Ergebnis liest sich noch gut, auch wenn es bei jedem Lauf anders aussieht. Zahlen („4/4“, „1960er“) und Satzzeichen sind unberührt, und der Anfangsbuchstabe jedes Wortes ist geblieben (auch Großbuchstaben). Die anderen Beispiele:

In [ ]:
random.seed(37)
for nummer in [2, 3, 4]:
    print('--- twist' + str(nummer) + '.txt')
    print(twiste_text(lies_text('beispieldaten/twist' + str(nummer) + '.txt')))
print('--- twist5.txt (Anfang)')
print(twiste_text(lies_text('beispieldaten/twist5.txt'))[:330])

--- twist2.txt
Hat der atle Hmxisneeeter sich doch eminal wgegbbeeen! Und nun sloeln siene Giseter acuh nach mienem Wellin lbeen. Sinee Wrot und Werke mrekt ich und den Barcuh, und mit Gärsessittkee tu ich Wnedur auch.
--- twist3.txt
Ein Rrautsneat, whceles a la crate aberiett, beeitt sien Aonebgt ohne eine vrehor fselegettge Mnfilhgüneoreee an. Dcrdauh haebn die Gtsäe zwar mher Slripeaum bei der Whal iehrr Ssiepen, für das Raurtnsaet enesethtn jcoedh zehiäuzcsltr Awfanud, da wngieer Pgnrlsucnheaisieht vdanorhen ist.
--- twist4.txt
Austgua Ada Byorn King, Centosus of Lvaoclee, war enie bithrisce Aildege und Mataihitekrmen, die als die ertse Prgrirmaoiermen üheubarpt glit. Beetirs 100 Jarhe vor dem Aukmmeofn der etresn Pmgmciepsrraeharorn esnran sie enie Rhceen-Meinhcak, der enigie Kozntepe menerdor Paamihmoprecrsrgren vgwanorehm.
--- twist5.txt (Anfang)
Alcie fnig an scih zu laelniwegn; sie saß shocn lngae bei irehr Shsecetwr am Ufer und httae nicths zu thun. Das Bcuh, das irhe Seswhtc

## Schritt 4: Das Wörterbuch aufbauen

Die Normalform eines Wortes: erster Buchstabe (klein geschrieben), dann die **sortierten** inneren Buchstaben, dann der letzte Buchstabe. Den ersten Buchstaben schreiben wir klein, weil ein Wort am Satzanfang großgeschrieben wird, in der Wörterliste aber vielleicht klein steht („Dsas“ am Satzanfang und „dass“ in der Liste sollen zusammenpassen).

Das Wörterbuch `INDEX` ordnet jeder Normalform die Liste der Wörter aus der Wörterliste zu.

In [ ]:
def normalform(wort):
    """Erster Buchstabe (klein) + sortierte innere Buchstaben + letzter Buchstabe."""
    if len(wort) <= 3:
        return wort.lower()                   # kurze Wörter werden nicht gemischt
    return wort[0].lower() + ''.join(sorted(wort[1:-1])) + wort[-1]


def baue_index(dateiname):
    index = {}
    with open(dateiname, encoding='utf-8') as datei:
        woerter = datei.read().split()
    for wort in woerter:
        schluessel = normalform(wort)
        if schluessel not in index:
            index[schluessel] = []
        if wort not in index[schluessel]:         # die Wörterliste enthält manche Wörter doppelt
            index[schluessel].append(wort)
    return index, len(woerter)


print(normalform('Programm'), normalform('Prarmgom'), normalform('Prrmgoam'))
INDEX, ANZAHL_WOERTER = baue_index('beispieldaten/woerterliste.txt')
print(ANZAHL_WOERTER, 'Wörter,', len(INDEX), 'verschiedene Normalformen')
mehrdeutig = [s for s in INDEX if len(INDEX[s]) > 1]
print('Normalformen mit mehr als einem Wort:', len(mehrdeutig))
print('Beispiele:', INDEX[normalform('Programm')], INDEX[normalform('Akteur')], INDEX[normalform('frühen')], INDEX[normalform('handelte')])

pagmorrm pagmorrm pagmorrm
312002 Wörter, 306057 verschiedene Normalformen
Normalformen mit mehr als einem Wort: 2728
Beispiele: ['Programm'] ['Akteur', 'akuter'] ['frühen', 'führen'] ['haltende', 'handelte']


Die Wörterliste hat 312 002 Einträge. Manche Wörter stehen darin doppelt, deshalb prüft `baue_index`, ob ein Wort schon in der Liste des Schlüssels steht. Es gibt 306 057 verschiedene Normalformen. Für fast alle Wörter ist das Enttwisten also eindeutig, nur bei 2 728 Normalformen gibt es mehrere Wörter, zum Beispiel `Akteur`/`akuter` oder `handelte`/`haltende`. (Ein Wort hat nur wenige Buchstaben, deshalb kostet das Sortieren wenig: Der Aufwand beim Aufbau ist ungefähr proportional zur Zahl der Wörter.)

## Schritt 5: Ein Wort enttwisten

Zu einem getwisteten Wort schlagen wir die Normalform nach und erhalten die Kandidaten. Dann ist noch die **Groß- und Kleinschreibung** zu klären:

1. Beginnt das getwistete Wort **groß**, kann es ein Hauptwort sein oder ein beliebiges Wort am Satzanfang. Wir machen bei allen Kandidaten den ersten Buchstaben groß: Aus „dass“ der Liste wird für „Dsas“ das „Dass“.
2. Beginnt es **klein**, ist es mitten im Satz, und ein Hauptwort kommt nicht in Frage. Wir nehmen nur kleingeschriebene Kandidaten, falls es welche gibt. Gibt es keine, machen wir den Anfangsbuchstaben klein (aus „Takt“ der Liste wird für „tkat“ das Wort „takt“).
3. Gibt es keinen Kandidaten, bleibt das Wort, wie es ist.

Das Ergebnis ist eine Liste (leer, wenn das Wort unbekannt ist). Doppelte Einträge lassen wir weg, denn nach der Anpassung der Schreibweise können zwei Kandidaten gleich aussehen.

In [ ]:
def kandidaten(wort):
    """Alle Wörter der Wörterliste, die ein Original von 'wort' sein können (leere Liste, wenn keines)."""
    if len(wort) <= 3:
        return [wort]
    treffer = INDEX.get(normalform(wort), [])
    if not wort[0].isupper():
        # Kleingeschrieben im Text: Hauptwörter (groß) kommen nicht in Frage, wenn es kleine Treffer gibt
        klein = [k for k in treffer if k[0] == wort[0]]
        if len(klein) > 0:
            return klein
    angepasst = []
    for k in treffer:
        if wort[0].isupper():
            k = k[0].upper() + k[1:]      # Großbuchstabe am Anfang: Haupt- oder Satzanfangswort
        else:
            k = k[0].lower() + k[1:]
        if k not in angepasst:
            angepasst.append(k)
    return angepasst


for wort in ['Prarmgom', 'frhüen', 'Dsas', 'Tkat', 'tkat', 'Mdaotenz']:
    print(wort, '->', kandidaten(wort))

Prarmgom -> ['Programm']
frhüen -> ['frühen', 'führen']
Dsas -> ['Dass']
Tkat -> ['Takt']
tkat -> ['takt']
Mdaotenz -> []


## Schritt 6: Einen ganzen Text enttwisten

Wie beim Twisten zerlegen wir den Text und setzen ihn wieder zusammen. Bei einem Wort gibt es drei Fälle:

- **ein Kandidat:** wir setzen ihn ein,
- **mehrere Kandidaten:** wir schreiben alle in eckigen Klammern, `[frühen,führen]`, damit ein Mensch entscheiden kann,
- **kein Kandidat:** das Wort bleibt unverändert (es steht nicht in der Wörterliste).

In [ ]:
def enttwiste_text(text):
    ergebnis = ''
    for ist_wort, stueck in zerlege(text):
        if not ist_wort:
            ergebnis = ergebnis + stueck
            continue
        k = kandidaten(stueck)
        if len(k) == 0:
            ergebnis = ergebnis + stueck                       # unbekanntes Wort
        elif len(k) == 1:
            ergebnis = ergebnis + k[0]
        else:
            ergebnis = ergebnis + '[' + ','.join(k) + ']'      # mehrdeutig
    return ergebnis


getwistet = lies_text('beispieldaten/enttwist.txt')
print(enttwiste_text(getwistet))

Der Twsit
(Englisch tiwst = Drehung, Verdrehung)
war ein Mdaotenz im 4/4-Takt,
der in den [frühen,führen] 1960er Jahren populär
wurde und zu
Rock'n'Roll, Ryhthm and Blues oder spezieller
Twsit-Musik getanzt wird.


Das stimmt mit den Lösungshinweisen überein, bis auf die Großschreibung (bei uns „Englisch“, weil das getwistete Wort groß beginnt). Nur vier Wörter bleiben offen:

- **„Mdaotenz“, „Ryhthm“, „Twsit“** (drei Wörter) bleiben stehen, weil die Wörter nicht in der Wörterliste stehen („Modetanz“, „Rhythm“ und „Twist“ fehlen). Ein Mensch errät sie aus dem Zusammenhang, das Programm kann es mit nur dieser Wörterliste nicht.
- **„frhüen“ ist mehrdeutig:** „frühen“ oder „führen“. Hier hilft nur der Zusammenhang („in den ... 1960er Jahren“). Das Programm zeigt beide Möglichkeiten.

Alles andere wird richtig zugeordnet, auch Wörter mit großem Anfangsbuchstaben wie „Eigsnclh“ (Englisch) oder „Duenrhg“ (Drehung).

## Schritt 7: Wie gut ist das Enttwisten?

Wir testen es mit den vier Texten zum Twisten: Jeden Text twisten wir (mit festem `seed`), enttwisten ihn mit unserem Programm und vergleichen Wort für Wort mit dem Original. Für jedes Wort mit mindestens 4 Buchstaben (kürzere ändert Twisten nicht) gibt es vier Ausgänge:

- **eindeutig richtig:** genau ein Kandidat, und der ist das Originalwort,
- **mehrdeutig:** mehrere Kandidaten, das Original ist dabei (ein Mensch oder ein Zusatzverfahren muss entscheiden),
- **unbekannt:** kein Kandidat (das Original steht nicht in der Wörterliste),
- **falsch:** ein einziger oder mehrere Kandidaten, aber das Original ist nicht dabei (ein anderes Wort der Liste hat dieselbe Normalform).

In [ ]:
def bewerte(dateiname, startwert):
    original = lies_text(dateiname)
    random.seed(startwert)
    getwistet = twiste_text(original)
    zaehler = {'richtig': 0, 'mehrdeutig': 0, 'unbekannt': 0, 'falsch': 0}
    unbekannte = []
    paare = list(zip(zerlege(original), zerlege(getwistet)))
    for (ist_wort, wort), (_, verdreht) in paare:
        if not ist_wort or len(wort) <= 3:
            continue
        k = kandidaten(verdreht)
        if len(k) == 0:
            zaehler['unbekannt'] += 1
            unbekannte.append(wort)
        elif wort not in k:
            zaehler['falsch'] += 1
        elif len(k) == 1:
            zaehler['richtig'] += 1
        else:
            zaehler['mehrdeutig'] += 1
    return zaehler, unbekannte


ERGEBNISSE = {}
print('Beispiel   Wörter  richtig  mehrdeutig  unbekannt  falsch')
for nummer in [2, 3, 4, 5]:
    zaehler, unbekannte = bewerte('beispieldaten/twist' + str(nummer) + '.txt', 37)
    ERGEBNISSE[nummer] = (zaehler, unbekannte)
    summe = sum(zaehler.values())
    print('twist' + str(nummer), '%8d %8d %11d %10d %7d' % (summe, zaehler['richtig'], zaehler['mehrdeutig'], zaehler['unbekannt'], zaehler['falsch']))

Beispiel   Wörter  richtig  mehrdeutig  unbekannt  falsch
twist2       22       21           0          1       0
twist3       29       25           1          3       0
twist4       27       21           1          5       0
twist5     2325     2081          52        189       3


Beobachtungen:

- Bei den deutschen Texten twist2 bis twist4 sind fast alle Wörter eindeutig richtig. Es bleiben wenige unbekannte Wörter: „wegbegeben“ (twist2), „carte“, „Menüreihenfolge“ und „Planungssicherheit“ (twist3), die Namen „Augusta“, „Byron“, „King“, „Countess“ und „Lovelace“ (twist4). Falsche Zuordnungen gibt es hier keine.
- Mehrdeutig sind nur zwei Wörter: „arbeitet“ (oder „abrietet“) in twist3 und „Bereits“ (oder „Bieters“, „Breites“) in twist4. Ein Mensch entscheidet aus dem Satz sofort. Am Satzanfang kann ein kleines Wort der Liste („bereits“) als großes erscheinen, deshalb kommen dort mehr Kandidaten in Frage.
- Bei `twist5.txt` (Alice im Wunderland, alte Rechtschreibung) gibt es viel mehr unbekannte Wörter (189 von 2325), weil Wörter wie „thun“, „werth“ oder „Thür“ in der heutigen Wörterliste fehlen und viele Zusammensetzungen nicht drin stehen. Hier gibt es auch die meisten Mehrdeutigkeiten, etwa „Garten“/„Graten“ oder „fiel“/„feil“, und drei falsche Zuordnungen („dunkeln“ wird zu „dunklen“, „hersagte“ zu „hagerste“, „genähret“ zu „genähert“): Das Originalwort steht nicht in der Liste, ein anderes mit derselben Normalform schon.
- Insgesamt wird in twist5 für knapp 90 % der Wörter (2081 von 2325) das Original eindeutig gefunden; bei weiteren 52 Wörtern steht es in der Kandidatenliste.

Die unbekannten Wörter von twist5 (Auswahl):

In [ ]:
unbekannte5 = ERGEBNISSE[5][1]
einmalig = []
for w in unbekannte5:
    if w not in einmalig:
        einmalig.append(w)
print(len(einmalig), 'verschiedene unbekannte Wörter, zum Beispiel:')
print(', '.join(einmalig[:20]))

138 verschiedene unbekannte Wörter, zum Beispiel:
thun, werth, rothen, vorbeirannte, fortlief, Grasplatz, hineingesprungen, Kaninchenbau, mußte, Küchenschränken, Vorbeifallen, Töpfchen, Verdruß, tödten, vorbeikam, muthig, Redens, Dachspitze, endigen, funfzig


## Für Neugierige: Mehrdeutigkeiten auflösen

Bei mehrdeutigen Wörtern gibt unser Programm alle Kandidaten aus. Ein Mensch entscheidet sofort aus dem Zusammenhang. Eine Maschine könnte dafür **Wortpaare** nutzen: Wie oft steht „den frühen“ in einem großen deutschen Text, wie oft „den führen“? Das braucht aber einen großen Text als zweite Datenquelle, den wir hier nicht haben. Auch Namen und Fremdwörter bleiben ein Problem: Man müsste die Wörterliste um sie ergänzen.

Zur Antwort auf die Frage der Aufgabe („Kann eine Maschine twisten, was ein Mensch lesen kann?“): Ja, **fast immer**, weil die Normalform einen Großteil der Information erhält (Anfangs- und Endbuchstabe, Länge, Buchstabenmenge). Twisten schützt also nur gegen oberflächliche Suche nach Wörtern, nicht gegen ein Programm mit Wörterliste. Wer es besser verstecken will, müsste auch Anfang und Ende mischen, doch dann werden auch Menschen langsamer.

Das Verfahren braucht pro Wort nur Sortieren der inneren Buchstaben und ein Nachschlagen. Das Wörterbuch aufzubauen kostet einmal etwa so viel wie das Durchlesen der Wörterliste.

## Zum Schluss: Twist mit MiniZinc?

Wir haben geprüft, ob sich das Enttwisten mit MiniZinc sinnvoll lösen lässt. Das naheliegende Modell: Für jedes getwistete Wort ist eine Variable `d[k]` der Index eines Wortes der Wörterliste, und Nebenbedingungen verlangen gleiche Länge, gleichen ersten und letzten Buchstaben und für jeden Buchstaben die gleiche Anzahl innerer Vorkommen. Dazu muss die **ganze** Wörterliste als Tabelle in die Datendatei (312 001 Zeilen mal 35 Buchstaben, rund 36 MB).

Gemessen mit Gecode am Beispiel `enttwist.txt` (nur 20 verschiedene Wörter mit mindestens 4 Buchstaben): Nach 100 Sekunden war noch keine Lösung gefunden. Dazu kommt, dass das Modell auch bei schneller Suche **unlösbar** wäre, sobald ein Wort nicht in der Wörterliste steht („Twsit“, „Mdaotenz“), und dass mehrdeutige Wörter nur *irgendeine* Lösung ergeben.

Deshalb haben wir kein MiniZinc-Modell aufgenommen:

- Das Problem hat keine Optimierung und keine Zuordnung unter Nebenbedingungen. Es ist reines **Nachschlagen**, und dafür ist das Wörterbuch (`dict`) mit der Normalform in Python wesentlich besser (ein Nachschlagen statt einer Suche über 312 001 Wörter).
- Die Wörterliste ist für MiniZinc viel zu groß, die Datendatei allein ist es schon.